# 1D J1J2 Inference: LorentzRNN (seed 111-555)

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('../../1_hypnqs_torch_lorentz/utility_lorentz')
from j1j2_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [8]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [9]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test_no_tau(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    with torch.no_grad():
        test_samples_after = wf.sample_no_tau(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
numsamples = 10000
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'results_LorentzRNN'

## J2=0.0. sc=4.0

In [5]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251
spatial_clamp=4.0

In [11]:
# Earlier training
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 172 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-44.014400482177734-0.00039999998989515007j), var E = 0.23739999532699585
DMRG energy  is -44.1277
Time taken=0.172 hrs


In [5]:
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.27159881591797-0.00039999998989515007j), var E = 2.35260009765625
DMRG energy  is -44.1277
Time taken=0.36 hrs


In [13]:
seed=555
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 89 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.20180130004883-0.0019000000320374966j), var E = 0.04230000078678131
DMRG energy  is -44.1277
Time taken=0.17 hrs


## J2=0.2, sc=2.0

In [15]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388
spatial_clamp =2.0

In [19]:
# Earlier training
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.35369873046875-9.999999747378752e-05j), var E = 0.26190000772476196
DMRG energy  is -40.7388
Time taken=0.384 hrs


In [20]:
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 646 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.38759994506836-9.999999747378752e-05j), var E = 0.07440000027418137
DMRG energy  is -40.7388
Time taken=0.39 hrs


In [16]:
seed=222
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 151 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.18659973144531-0.000699999975040555j), var E = 0.8481000065803528
DMRG energy  is -40.7388
Time taken=0.294 hrs


In [17]:
seed=333
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 279 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.5609016418457+0.0007999999797903001j), var E = 0.2224999964237213
DMRG energy  is -40.7388
Time taken=0.361 hrs


In [18]:
seed=444
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 383 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.464599609375-0.0007999999797903001j), var E = 0.3402000069618225
DMRG energy  is -40.7388
Time taken=0.393 hrs


## J2=0.5, sc=2.0

Seed 111 didn't converge, replaced by seed 666.

In [4]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=-37.5000
spatial_clamp=2.0

In [6]:
seed=222
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 36 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.08140182495117+0.0005000000237487257j), var E = 1.7833000421524048
DMRG energy  is 37.5
Time taken=0.489 hrs


In [13]:
# Seed 333: Best model saved at epoch 955 with best E=-37.31641-0.27781j, varE=15.86059
seed=333
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.60580062866211-0.0038999998942017555j), var E = 2.7276999950408936
DMRG energy  is 37.5
Time taken=0.466 hrs


In [9]:
seed=444
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 264 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.860198974609375-0.0003000000142492354j), var E = 0.6093999743461609
DMRG energy  is 37.5
Time taken=0.525 hrs


In [15]:
# Seed 555: Best model saved at epoch 843 with best E=-37.46337+0.08807j, varE=11.37121
seed=555
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.142799377441406-0.0020000000949949026j), var E = 0.21220000088214874
DMRG energy  is 37.5
Time taken=0.495 hrs


In [6]:
#Seed 666: Best model saved at epoch 934 with best E=-37.62999+0.00121j, varE=0.85362
seed=666
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 39 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.86709976196289+0.0015999999595806003j), var E = 1.5293999910354614
DMRG energy  is 37.5
Time taken=0.179 hrs


## J2=0.8
 Seeds 333, 555: No convergence

In [ ]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643
spatial_clamp=2.0

In [5]:
seed=111
set_cpu_deterministic(seed)
spatial_clamp=2.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 424 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.18730163574219+0.0024999999441206455j), var E = 2.357800006866455
DMRG energy  is -42.0701
Time taken=0.75 hrs


In [6]:
seed=222
set_cpu_deterministic(seed)
spatial_clamp=2.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
Clipped 156 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.33300018310547-0.0071000000461936j), var E = 1.2055000066757202
DMRG energy  is -42.0701
Time taken=0.594 hrs


In [9]:
seed=444
set_cpu_deterministic(seed)
spatial_clamp=2.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_N=100_LorentzRNN_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,395
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.956298828125+0.004999999888241291j), var E = 5.971499919891357
DMRG energy  is -42.0701
Time taken=0.298 hrs
